# Notebook 02 - Train Agent B (ViT-B/16) on Kaggle GPU

Argus Vision: adversarial multi-agent visual debate for uncertainty-aware medical image classification.

This notebook trains **Agent B**, a **ViT-B/16** transformer classifier from `timm`
(`vit_base_patch16_224.augreg_in21k_ft_in1k`), on the ISIC-2019 8-class dermoscopy
taxonomy: `["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]`.

## Kaggle setup (do this before Run All)
1. **Attach the dataset**: Add Data -> search `andrewmvd/isic-2019` -> Add. It mounts read-only at `/kaggle/input/isic-2019`.
2. **Accelerator**: Settings -> Accelerator -> **GPU T4 x2** (or **P100**).
3. **Internet**: Settings -> Internet -> **ON** (needed for `pip install` and pretrained ViT weights).
4. **Downstream notebooks**: later notebooks (build hard subset, consensus, evaluation) should additionally attach **this notebook's output** as an input dataset to reuse `agent_b_best.pth`.
5. **Run All**.

Pipeline: env setup + dataset discovery -> EDA -> dataset/sampler/loaders -> ViT-B/16 -> Phase 1 head-only -> Phase 2 fine-tune last 4 blocks (attention layers at 0.1x LR) -> curves -> save best -> confusion matrix -> **attention rollout** visualization. The notebook is fully self-contained (no sibling-module imports).

## 1. Environment setup, shared helpers, and dataset discovery

Install the extras this notebook needs (`timm -U`, `grad-cam`, `torchmetrics`), define the
shared Argus constants/helpers (`ISIC_CLASSES`, `discover_isic`, `find_file`), and locate
the ISIC-2019 ground-truth CSV + image directory under `/kaggle/input`.

In [ ]:
# --- Install extras (Kaggle preinstalls torch/torchvision/numpy/pandas/sklearn/etc.) ---
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm",
                "grad-cam", "torchmetrics"])
print("NOTE: Internet must be ON (Settings -> Internet -> ON) for pip + pretrained ViT weights.")

import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Argus shared constants (canonical ISIC-8 contract) ---
ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
FULL_NAMES = {
    "MEL": "Melanoma", "NV": "Melanocytic nevus", "BCC": "Basal cell carcinoma",
    "AK": "Actinic keratosis", "BKL": "Benign keratosis", "DF": "Dermatofibroma",
    "VASC": "Vascular lesion", "SCC": "Squamous cell carcinoma",
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8
SEED = 42
BATCH_SIZE = 32          # lower to 16 if you hit CUDA OOM on T4
NUM_WORKERS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def discover_isic(root="/kaggle/input"):
    """Walk /kaggle/input and return (csv_path, image_dir) for ISIC-2019.

    The ground-truth CSV is the .csv whose header contains ALL 8 class names.
    The image dir is the directory holding the most ISIC_*.jpg/.jpeg files.
    Robust to nested / double-nested folders.
    """
    csv_path = None
    needed = set(ISIC_CLASSES)
    image_counts = {}
    for dirpath, _dirnames, filenames in os.walk(root):
        jpg_here = 0
        for fn in filenames:
            low = fn.lower()
            if csv_path is None and low.endswith(".csv"):
                try:
                    cols = set(pd.read_csv(os.path.join(dirpath, fn), nrows=0).columns)
                except Exception:
                    cols = set()
                if needed.issubset(cols):
                    csv_path = os.path.join(dirpath, fn)
            if low.startswith("isic_") and (low.endswith(".jpg") or low.endswith(".jpeg")):
                jpg_here += 1
        if jpg_here:
            image_counts[dirpath] = jpg_here
    image_dir = max(image_counts, key=image_counts.get) if image_counts else None
    print("Discovered CSV      :", csv_path)
    print("Discovered image dir:", image_dir,
          f"({image_counts.get(image_dir, 0):,} images)" if image_dir else "")
    assert csv_path is not None, "Could not find an ISIC GroundTruth CSV with all 8 class columns."
    assert image_dir is not None, "Could not find an ISIC image directory with ISIC_*.jpg files."
    return csv_path, image_dir


def find_file(filename_substring, search_roots=("/kaggle/input", "/kaggle/working")):
    """Return the first path whose basename contains the substring, else None.

    Used to locate previously-trained .pth checkpoints / hard_subset.csv across
    any attached datasets.
    """
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                if filename_substring in fn:
                    return os.path.join(dirpath, fn)
    print(f"find_file: no file containing {filename_substring!r} under {search_roots}")
    return None


CSV_PATH, IMAGE_DIR = discover_isic()
WORK_DIR = "/kaggle/working"
os.makedirs(os.path.join(WORK_DIR, "figures"), exist_ok=True)
CHECKPOINT_PATH = os.path.join(WORK_DIR, "agent_b_best.pth")

## 2. Exploratory data analysis

Load the ISIC-2019 one-hot ground-truth CSV, collapse the class columns into the canonical
ISIC-8 integer label (argmax over the 8 columns), plot the class distribution and show a
3x3 grid of representative sample images.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

raw = pd.read_csv(CSV_PATH)
# Identify the image-id column (a column named 'image', else the first column).
img_col = "image" if "image" in raw.columns else raw.columns[0]
# label = argmax over the 8 canonical class columns (ignore UNK if present).
label_matrix = raw[ISIC_CLASSES].to_numpy(dtype=np.float32)
labels = label_matrix.argmax(axis=1).astype(np.int64)

df = pd.DataFrame({
    "image": raw[img_col].astype(str).values,
    "label": labels,
})
df["path"] = df["image"].apply(lambda s: os.path.join(IMAGE_DIR, s + ".jpg"))
# Keep only rows whose image file actually exists on disk.
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Loaded {len(df):,} labelled images with existing files.")
df.head()

In [ ]:
# Class-distribution bar chart over the eight ISIC classes.
counts = df["label"].value_counts().sort_index()
class_counts = pd.Series(
    [int(counts.get(idx, 0)) for idx in range(NUM_CLASSES)],
    index=ISIC_CLASSES,
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax,
            hue=class_counts.index, palette="mako", legend=False)
ax.set_title("ISIC-8 class distribution (full dataset)")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
for patch, value in zip(ax.patches, class_counts.values):
    ax.annotate(f"{int(value):,}",
                (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("Per-class counts:")
for name, value in class_counts.items():
    print(f"  {name:5s} ({FULL_NAMES[name]:25s}): {int(value):,}")

In [ ]:
# 3x3 grid: one representative sample image per class (wraps if <9 classes).
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax_idx, ax in enumerate(axes.ravel()):
    class_idx = ax_idx % NUM_CLASSES
    subset = df[df["label"] == class_idx]
    if len(subset) == 0:
        ax.axis("off")
        continue
    row = subset.iloc[int(rng.integers(0, len(subset)))]
    img = Image.open(row["path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{ISIC_CLASSES[class_idx]} - {FULL_NAMES[ISIC_CLASSES[class_idx]]}", fontsize=10)
    ax.axis("off")
plt.suptitle("ISIC-8 sample dermoscopic images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Dataset, samplers and DataLoaders

Inline `ISICDataset` (returns `(tensor, label_int, image_path)`), build a stratified
85/15 train/val split, apply the augreg-style ViT train/val transforms, compute
**inverse-sqrt-frequency** class weights (normalized to mean 1) and wire a
`WeightedRandomSampler` so minority classes (DF, VASC, SCC) are over-sampled.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms


class ISICDataset(Dataset):
    """Reads the one-hot ISIC-2019 GroundTruth dataframe.

    __getitem__ returns (image_tensor, label_int, image_path).
    """

    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = row["path"]
        label = int(row["label"])
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label, path


def build_train_transforms(size=IMAGE_SIZE):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.1),
    ])


def build_val_transforms(size=IMAGE_SIZE):
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


# Stratified 85/15 split preserves per-class ratios across train and validation.
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

train_dataset = ISICDataset(train_df, transform=build_train_transforms())
val_dataset = ISICDataset(val_df, transform=build_val_transforms())

In [ ]:
# Inverse-sqrt-frequency class weights, normalized to mean 1.
train_labels = train_df["label"].to_numpy()
class_sample_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float64)
class_sample_counts = np.clip(class_sample_counts, 1.0, None)

inv_sqrt = 1.0 / np.sqrt(class_sample_counts)
class_weights = inv_sqrt / inv_sqrt.mean()  # mean == 1
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights (inv-sqrt, mean 1):",
      {name: round(float(w), 3) for name, w in zip(ISIC_CLASSES, class_weights)})

# Per-sample weights drive the WeightedRandomSampler to balance minibatches.
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

pin = (DEVICE == "cuda")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 4. Model: ViT-B/16 with a frozen backbone

Create an ImageNet-21k -> 1k fine-tuned ViT-B/16 (`vit_base_patch16_224.augreg_in21k_ft_in1k`)
with an 8-class head via `timm`. Freeze the backbone so Phase 1 trains only the head.

In [ ]:
import timm

# timm ViT-B/16; the -U install above ensures this model id resolves.
MODEL_NAME = "vit_base_patch16_224.augreg_in21k_ft_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# Freeze everything, then unfreeze only the classifier head for Phase 1.
for p in model.parameters():
    p.requires_grad = False
for p in model.get_classifier().parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params (head only): {trainable:,} / {total:,}")
print("Num transformer blocks:", len(model.blocks))

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from torchmetrics.classification import MulticlassAUROC


class FocalLoss(nn.Module):
    """Multi-class focal loss with optional per-class alpha weighting.

    loss = (1 - p_t) ** gamma * CE, where p_t is the prob of the true class.
    """

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer("alpha", alpha)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = (1.0 - pt) ** self.gamma * ce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


auroc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average="macro").to(DEVICE)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


@torch.no_grad()
def evaluate(network, loader):
    """Return (balanced_accuracy, macro_auc, y_true, y_pred)."""
    network.eval()
    auroc_metric.reset()
    all_preds, all_targets = [], []
    for images, targets, _paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            probs = F.softmax(network(images), dim=1)
        auroc_metric.update(probs, targets)
        all_preds.append(probs.argmax(dim=1).detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())
    preds_arr = np.concatenate(all_preds)
    targets_arr = np.concatenate(all_targets)
    bal_acc = float(balanced_accuracy_score(targets_arr, preds_arr))
    macro_auc = float(auroc_metric.compute().item())
    return bal_acc, macro_auc, targets_arr, preds_arr


def run_epoch(network, loader, criterion, optimizer):
    """Run one mixed-precision optimization epoch; return mean train loss."""
    network.train()
    running_loss, num_batches = 0.0, 0
    for images, targets, _paths in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = network(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())
        num_batches += 1
    return running_loss / max(num_batches, 1)


history = {"loss": [], "bal_acc": [], "macro_auc": []}
best_macro_auc = -1.0

## 5. Phase 1 - head-only training

Train only the classifier head with `FocalLoss(alpha=class_weights, gamma=2.0)` and
`AdamW(lr=1e-3)`. Each epoch logs validation balanced accuracy + macro one-vs-rest AUC
and snapshots the best checkpoint to `/kaggle/working/agent_b_best.pth`.

In [ ]:
MAX_EPOCHS_HEAD = 5
MAX_EPOCHS_FINETUNE = 15
WEIGHT_DECAY = 1e-4
LR_HEAD = 1e-3
BASE_LR = 1e-5            # Phase 2 base LR
ATTN_LR_SCALE = 0.1      # attention-layer params train at base_lr * 0.1
FOCAL_GAMMA = 2.0

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=class_weights_tensor).to(DEVICE)

head_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
)

for epoch in range(1, MAX_EPOCHS_HEAD + 1):
    train_loss = run_epoch(model, train_loader, criterion, head_optimizer)
    bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
    history["loss"].append(train_loss)
    history["bal_acc"].append(bal_acc)
    history["macro_auc"].append(macro_auc)
    print(f"[P1 {epoch:02d}/{MAX_EPOCHS_HEAD}] "
          f"loss={train_loss:.4f} bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f}")
    if not math.isnan(macro_auc) and macro_auc > best_macro_auc:
        best_macro_auc = macro_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"    new best macro AUC={best_macro_auc:.4f} -> saved {CHECKPOINT_PATH}")

## 6. Phase 2 - fine-tune the last 4 transformer blocks

Unfreeze `model.blocks[-4:]` + the final `norm` + the head. Build **two** AdamW param
groups: attention-layer params (names containing `.attn.`) train at `base_lr * 0.1`,
everything else at `base_lr` (`1e-5`). Use `CosineAnnealingLR` and AMP. The best
checkpoint by macro AUC continues to be tracked across both phases.

In [ ]:
# Freeze all, then unfreeze last 4 blocks + final norm + head.
for p in model.parameters():
    p.requires_grad = False
for blk in list(model.blocks)[-4:]:
    for p in blk.parameters():
        p.requires_grad = True
if hasattr(model, "norm") and isinstance(model.norm, nn.Module):
    for p in model.norm.parameters():
        p.requires_grad = True
for p in model.get_classifier().parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (last 4 blocks + norm + head): {trainable:,}")

# Two param groups: attention-layer params (name has '.attn.') at base_lr*0.1,
# everything else at base_lr.
attn_params, other_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if ".attn." in name:
        attn_params.append(p)
    else:
        other_params.append(p)
print(f"attn params: {sum(p.numel() for p in attn_params):,} | "
      f"other params: {sum(p.numel() for p in other_params):,}")

finetune_optimizer = torch.optim.AdamW(
    [
        {"params": attn_params, "lr": BASE_LR * ATTN_LR_SCALE},
        {"params": other_params, "lr": BASE_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    finetune_optimizer, T_max=MAX_EPOCHS_FINETUNE,
)

for epoch in range(1, MAX_EPOCHS_FINETUNE + 1):
    train_loss = run_epoch(model, train_loader, criterion, finetune_optimizer)
    scheduler.step()
    bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
    history["loss"].append(train_loss)
    history["bal_acc"].append(bal_acc)
    history["macro_auc"].append(macro_auc)
    print(f"[P2 {epoch:02d}/{MAX_EPOCHS_FINETUNE}] "
          f"loss={train_loss:.4f} bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f}")
    if not math.isnan(macro_auc) and macro_auc > best_macro_auc:
        best_macro_auc = macro_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"    new best macro AUC={best_macro_auc:.4f} -> saved {CHECKPOINT_PATH}")

print(f"Best macro AUC across both phases: {best_macro_auc:.4f}")

## 7. Training curves

Plot training loss, validation balanced accuracy and validation macro AUC across the
combined Phase 1 + Phase 2 schedule. The dashed line marks the P1 -> P2 boundary.

In [ ]:
epochs_axis = np.arange(1, len(history["loss"]) + 1)
phase_boundary = MAX_EPOCHS_HEAD + 0.5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
panels = [
    ("loss", "Training loss", "tab:red"),
    ("bal_acc", "Val balanced accuracy", "tab:blue"),
    ("macro_auc", "Val macro AUC", "tab:green"),
]
for ax, (key, title, color) in zip(axes, panels):
    ax.plot(epochs_axis, history[key], marker="o", color=color)
    ax.axvline(phase_boundary, linestyle="--", color="gray", alpha=0.7, label="P1->P2")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
plt.suptitle("Agent B (ViT-B/16) training curves", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_curves.png"), dpi=120, bbox_inches="tight")
plt.show()

## 8. Reload best checkpoint

Reload the best weights observed during training (already serialized to
`/kaggle/working/agent_b_best.pth` whenever macro AUC improved) for the downstream
confusion matrix and attention-rollout visualizations.

In [ ]:
best_state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_state)
model.eval()
print(f"Reloaded best Agent B checkpoint from {CHECKPOINT_PATH}")
print(f"File size: {os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB")

## 9. Confusion matrix

Run the best model over the validation split and render a row-normalized per-class
confusion-matrix heatmap.

In [ ]:
from sklearn.metrics import confusion_matrix

_, _, y_true, y_pred = evaluate(model, val_loader)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, np.clip(row_sums, 1, None), dtype=np.float64)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="cividis",
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES, ax=ax,
            cbar_kws={"label": "Row-normalized frequency"})
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Agent B - validation confusion matrix (row-normalized)")
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_confusion.png"), dpi=120, bbox_inches="tight")
plt.show()

## 10. Attention rollout visualization

ViTs do not have a conv feature map, so instead of Grad-CAM we use **attention rollout**
(Abnar & Zuidema, 2020). We disable fused attention, register a forward hook on every
`block.attn` to capture its input, recompute the per-head softmax attention from the
block's `qkv`, average heads, form `A_hat = 0.5*(A + I)`, row-normalize, and multiply
across all layers. The CLS row over the 196 patch tokens is reshaped to 14x14, upsampled
to 224x224 and overlaid on a 3x3 grid of validation images. Hooks are removed in a
`finally` block.

In [ ]:
import cv2

model.eval()
mean = np.array(IMAGENET_MEAN, dtype=np.float32)
std = np.array(IMAGENET_STD, dtype=np.float32)

# Disable timm fused attention so we can recompute per-head softmax weights ourselves.
for blk in model.blocks:
    if hasattr(blk.attn, "fused_attn"):
        blk.attn.fused_attn = False

# Geometry: ViT-B/16 at 224 => 14x14 = 196 patch tokens + 1 CLS = 197 tokens.
GRID = IMAGE_SIZE // 16  # 14
NUM_PATCH_TOKENS = GRID * GRID

captured = []  # one captured (N, T, T) attention matrix per block, in forward order


def make_hook(attn_module):
    def hook(module, inputs, output):
        x = inputs[0]                      # (N, T, C) input to the attention block
        N, T, C = x.shape
        num_heads = int(module.num_heads)
        head_dim = C // num_heads
        scale = getattr(module, "scale", None)
        if scale is None:
            scale = head_dim ** -0.5
        qkv = module.qkv(x).reshape(N, T, 3, num_heads, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)   # (3, N, heads, T, head_dim)
        q, k = qkv[0], qkv[1]
        attn = (q @ k.transpose(-2, -1)) * float(scale)
        attn = attn.softmax(dim=-1)        # (N, heads, T, T)
        captured.append(attn.mean(dim=1).detach().to(torch.float32).cpu())  # avg heads
    return hook


def attention_rollout(input_tensor):
    """Return (224x224 heatmap in [0,1], predicted_class_index) for one image tensor."""
    captured.clear()
    with torch.no_grad():
        logits = model(input_tensor)
        pred_idx = int(torch.softmax(logits, dim=1).argmax(dim=1).item())
    # Multiply A_hat = 0.5*(A + I), row-normalized, across all layers.
    T = captured[0].shape[-1]
    eye = torch.eye(T, dtype=torch.float32)
    result = eye.clone()
    for attn in captured:
        a_hat = 0.5 * attn[0] + 0.5 * eye
        a_hat = a_hat / a_hat.sum(dim=-1, keepdim=True).clamp_min(1e-12)
        result = a_hat @ result
    # CLS row (token 0) over the 196 patch tokens (drop the CLS self-attention column).
    cls_attn = result[0, 1:][:NUM_PATCH_TOKENS]
    grid = cls_attn.reshape(GRID, GRID).numpy().astype(np.float32)
    heat = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC).astype(np.float32)
    span = float(heat.max() - heat.min())
    if span < 1e-12:
        heat = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
    else:
        heat = (heat - heat.min()) / span
    return np.clip(heat, 0.0, 1.0).astype(np.float32), pred_idx


grid_rows = val_df.sample(n=9, random_state=SEED).reset_index(drop=True)
handles = []
try:
    for blk in model.blocks:
        handles.append(blk.attn.register_forward_hook(make_hook(blk.attn)))

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for ax, (_, row) in zip(axes.ravel(), grid_rows.iterrows()):
        pil = Image.open(row["path"]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
        rgb = np.asarray(pil, dtype=np.float32) / 255.0
        norm = (rgb - mean) / std
        input_tensor = torch.from_numpy(norm.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
        heat, pred_idx = attention_rollout(input_tensor)
        heatmap = cv2.applyColorMap((heat * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        overlay = np.clip(0.55 * rgb + 0.45 * heatmap, 0.0, 1.0)
        ax.imshow(overlay)
        true_idx = int(row["label"])
        ax.set_title(f"true={ISIC_CLASSES[true_idx]} / pred={ISIC_CLASSES[pred_idx]}",
                     fontsize=10, color=("green" if pred_idx == true_idx else "red"))
        ax.axis("off")
    plt.suptitle("Agent B - attention rollout (CLS -> patch)", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, "figures", "agent_b_rollout.png"), dpi=120, bbox_inches="tight")
    plt.show()
finally:
    for h in handles:
        h.remove()
    print("Removed all attention hooks.")

## Outputs and next steps

All writable artifacts are under **`/kaggle/working/`**:
- `agent_b_best.pth` - best ViT-B/16 weights (by validation macro AUC). **This is the key output.**
- `figures/agent_b_curves.png`, `figures/agent_b_confusion.png`, `figures/agent_b_rollout.png`.

**Download:** In the Kaggle editor, open the right-hand **Output** panel and download
`agent_b_best.pth` (and the figures).

**Attach to the next notebook:** Save a version of this notebook (Save & Run All / Commit) so
the outputs are published. In the downstream notebooks (03 build hard subset, 04 consensus,
05 evaluation), click **Add Data -> Notebook Output** and select this notebook's output.
The provided `find_file("agent_b_best")` helper will then locate `agent_b_best.pth` automatically
under `/kaggle/input/...`. Remember to keep **Internet ON** and the **GPU** accelerator enabled
for those runs as well.